# Section 7 – Ranking Transitions and Institutional Mobility (Revised)

Tracks institution rank changes, top-10 membership, and movements between NIRF rank bands.

**Revisions:** exact ranks and published rank bands are both retained for transition analysis. Band positions use the published band midpoint only as an explicitly labelled estimate; no exact rank is fabricated. Unmatched/new/exit institutions are reported separately. The top-10 entry/exit trend is shown across all available years, avoiding an uninformative single-year zero chart. Rank-group transitions use real bands through 200 and a Not ranked/Outside published bands category. Interactive Plotly charts are used.

## Import Required Libraries

We use Pandas for data manipulation, NumPy for numerical operations, and Matplotlib for visualizations.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt

def load_master(path=None):
    """Load the NIRF master CSV from a supplied path or common Colab/local locations."""
    candidates = ([Path(path)] if path else []) + [
        Path("nirf_master_final.csv"),
        Path("/content/nirf_master_final.csv"),
        Path("/mnt/data/nirf_master_final.csv")
    ]
    for candidate in candidates:
        if candidate.exists():
            data = pd.read_csv(candidate)
            print(f"Loaded: {candidate} | shape={data.shape}")
            return data
    raise FileNotFoundError(
        "nirf_master_final.csv not found. Upload it to Colab or pass its full path to load_master(path)."
    )

def filter_data(data, **filters):
    """Reusable exact-value filters; e.g. filter_data(df, year=2025, category='Engineering')."""
    result = data.copy()
    for column, value in filters.items():
        if column not in result.columns:
            raise KeyError(f"Filter column '{column}' is not present.")
        result = result[result[column] == value]
    return result.copy()

df = load_master()

Loaded: /mnt/data/nirf_master_final.csv | shape=(11012, 42)


##  Load the NIRF Dataset

We load the final cleaned NIRF dataset containing institution details, rankings, categories, and years.

In [2]:
# Dataset is loaded by load_master() in the previous cell.
print("Rows:", len(df), "| Columns:", len(df.columns))
display(df.head())

Rows: 11012 | Columns: 42


,institution_key,institute_name,institute_id,institute_name_original,year,category,category_original,category_group,city,city_original,state,state_code,state_original,rank,rank_value,rank_status,rank_original,rank_low,rank_high,is_rank_band,rank_band_label,rank_band_source,rank_midpoint,score,TLR,RPC,GO,OI,PR,weight_scheme,is_top10,is_top50,is_top100,is_new_entry,first_year_in_category,category_first_year,categories_in_year,source_page,weight_scheme_basis,has_score,has_components,score_comparison_group
0,INST0936,Miranda House,IR17-COLL-1-29115,Miranda House,2017,College,College,General,Delhi,Delhi,Delhi,DL,Delhi,1.0,1.0,exact,1,1.0,1.0,False,NaN,NaN,1.0,69.39,75.76,27.44,90.06,96.45,41.82,30/20/25/15/10,True,True,True,False,2017,2017,6,CollegeRanking,estimated_from_data,True,True,College | 30/20/25/15/10
1,INST0814,Loyola College,IR17-COLL-2-959129,Loyola College,2017,College,College,General,Chennai,District Chennai,Tamil Nadu,TN,Tamil Nadu,2.0,2.0,exact,2,2.0,2.0,False,NaN,NaN,2.0,68.68,66.92,100.00,48.42,56.45,80.45,30/20/25/15/10,True,True,True,False,2017,2017,6,CollegeRanking,estimated_from_data,True,True,College | 30/20/25/15/10
2,INST1313,Shri Ram College of Commerce,IR17-COLL-2-29134,Shri Ram College of Commerce,2017,College,College,General,Delhi,Delhi,Delhi,DL,Delhi,3.0,3.0,exact,3,3.0,3.0,False,NaN,NaN,3.0,67.18,77.63,0.20,97.51,93.80,53.98,30/20/25/15/10,True,True,True,False,2017,2017,6,CollegeRanking,estimated_from_data,True,True,College | 30/20/25/15/10
3,INST0172,Bishop Heber College,IR17-COLL-1-19387,Bishop Heber College,2017,College,College,General,Tiruchirappalli,District Tiruchirapalli,Tamil Nadu,TN,Tamil Nadu,4.0,4.0,exact,4,4.0,4.0,False,NaN,NaN,4.0,61.18,76.62,53.99,61.28,68.11,18.63,30/20/25/15/10,True,True,True,False,2017,2017,6,CollegeRanking,estimated_from_data,True,True,College | 30/20/25/15/10
4,INST0093,Atma Ram Sanatan Dharma College,IR17-COLL-2-21799,Atma Ram Sanatan Dharma College,2017,College,College,General,New Delhi,New Delhi,Delhi,DL,Delhi,5.0,5.0,exact,5,5.0,5.0,False,NaN,NaN,5.0,60.68,74.22,30.54,79.60,82.51,0.21,30/20/25/15/10,True,True,True,False,2017,2017,6,CollegeRanking,estimated_from_data,True,True,College | 30/20/25/15/10


##  Understand the Dataset

We examine the columns, data types, available years, categories, and rank status values before performing the analysis.

In [3]:
print("Dataset information:")
df.info()

print("\nAvailable years:")
print(sorted(df["year"].dropna().unique()))

print("\nAvailable categories:")
print(sorted(df["category"].dropna().unique()))

print("\nRank status:")
print(df["rank_status"].value_counts(dropna=False))

Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11012 entries, 0 to 11011
Data columns (total 42 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   institution_key          11012 non-null  object 
 1   institute_name           11012 non-null  object 
 2   institute_id             6966 non-null   object 
 3   institute_name_original  11012 non-null  object 
 4   year                     11012 non-null  int64  
 5   category                 11012 non-null  object 
 6   category_original        11012 non-null  object 
 7   category_group           11012 non-null  object 
 8   city                     11012 non-null  object 
 9   city_original            11002 non-null  object 
 10  state                    11012 non-null  object 
 11  state_code               11012 non-null  object 
 12  state_original           11012 non-null  object 
 13  rank                     6954 non-null   float64
 14  r

##  Select the Category and Comparison Years

We select one category and two consecutive years.

The values can be changed to analyse ranking transitions for another category or year.

In [4]:
selected_category = "Engineering"

previous_year = 2024
current_year = 2025

print("Selected category:", selected_category)
print("Previous year:", previous_year)
print("Current year:", current_year)

Selected category: Engineering
Previous year: 2024
Current year: 2025


##  Prepare the Ranking Data

We create reusable functions to prepare institutional ranking data.

The functions:

- Select the required category and year.
- Keep only exact-ranked institutions.
- Convert rank values into numeric format.
- Remove missing institution identifiers and ranks.
- Keep one record per institution in each year.

Rank 1 represents the best position. Therefore, a decrease in the numerical rank indicates an improvement.

In [5]:
required_columns = ["institution_key","institute_name","year","category","rank","rank_status",
                    "rank_low","rank_high","rank_midpoint","rank_band_label"]
missing_columns = [c for c in required_columns if c not in df.columns]
if missing_columns:
    raise ValueError(f"Missing columns: {missing_columns}")

def prepare_year_rank_data(data, year, category, include_bands=False):
    result = filter_data(data, year=year, category=category).copy()
    status = result["rank_status"].astype(str).str.strip().str.lower()
    result = result[status.eq("exact") | (status.eq("band") if include_bands else False)].copy()
    for col in ["rank","rank_low","rank_high","rank_midpoint"]:
        result[col] = pd.to_numeric(result[col], errors="coerce")
    result = result.dropna(subset=["institution_key","institute_name"])
    result["rank_estimate"] = np.where(result["rank_status"].astype(str).str.lower().eq("exact"),
                                       result["rank"], result["rank_midpoint"])
    result["rank_group"] = np.where(
        result["rank_status"].astype(str).str.lower().eq("band"),
        result["rank_band_label"].fillna("Published band"),
        pd.NA
    )
    result.loc[result["rank_status"].astype(str).str.lower().eq("exact"), "rank_group"] = result.loc[
        result["rank_status"].astype(str).str.lower().eq("exact"), "rank"
    ].apply(lambda r: "Top 10" if r <= 10 else ("11-50" if r <= 50 else ("51-100" if r <= 100 else "Outside published bands")))
    if result.duplicated("institution_key").any():
        print(f"Warning: duplicate keys in {year}; retaining most informative record.")
        result["_quality"] = result["rank"].notna().astype(int) + result["rank_midpoint"].notna().astype(int)
        result = result.sort_values("_quality").drop_duplicates("institution_key",keep="last").drop(columns="_quality")
    return result[["institution_key","institute_name","rank","rank_status","rank_low","rank_high",
                   "rank_midpoint","rank_estimate","rank_group","rank_band_label"]]

previous_data = prepare_year_rank_data(df, previous_year, selected_category, include_bands=True)
current_data = prepare_year_rank_data(df, current_year, selected_category, include_bands=True)
previous_exact = previous_data[previous_data["rank_status"].astype(str).str.lower().eq("exact")].copy()
current_exact = current_data[current_data["rank_status"].astype(str).str.lower().eq("exact")].copy()
print("Previous year exact/band records:", len(previous_data))
print("Current year exact/band records:", len(current_data))

Previous year exact/band records: 200
Current year exact/band records: 201


## Match institutions across years

Exact-to-exact rank changes are calculated only when both years have exact ranks. A separate band-aware transition dataset includes exact ranks and published rank bands. Band midpoints are estimates for visual ordering only, not exact ranks.

In [6]:
comparison = previous_exact[["institution_key","institute_name","rank"]].rename(
    columns={"institute_name":"previous_name","rank":"previous_rank"}
).merge(
    current_exact[["institution_key","institute_name","rank"]].rename(
        columns={"institute_name":"current_name","rank":"current_rank"}
    ), on="institution_key", how="inner"
)
comparison["rank_improvement"] = comparison["previous_rank"] - comparison["current_rank"]
comparison["rank_movement"] = np.select(
    [comparison["rank_improvement"]>0, comparison["rank_improvement"]<0],
    ["Improved","Declined"], default="Maintained"
)
comparison["institute_name"] = comparison["current_name"]

previous_keys, current_keys = set(previous_data.institution_key), set(current_data.institution_key)
unmatched_previous = previous_data[~previous_data.institution_key.isin(current_keys)].copy()
unmatched_current = current_data[~current_data.institution_key.isin(previous_keys)].copy()
print("Exact-to-exact matched:", len(comparison))
print("Present previous year only:", len(unmatched_previous))
print("Present current year only:", len(unmatched_current))
print("These unmatched records are not included in exact rank movement percentages.")
display(comparison.head())

Exact-to-exact matched: 93
Present previous year only: 9
Present current year only: 10
These unmatched records are not included in exact rank movement percentages.


,institution_key,previous_name,previous_rank,current_name,current_rank,rank_improvement,rank_movement,institute_name
0,INST0585,Indian Institute of Technology Madras,1.0,Indian Institute of Technology Madras,1.0,0.0,Maintained,Indian Institute of Technology Madras
1,INST0574,Indian Institute of Technology Delhi,2.0,Indian Institute of Technology Delhi,2.0,0.0,Maintained,Indian Institute of Technology Delhi
2,INST0573,Indian Institute of Technology Bombay,3.0,Indian Institute of Technology Bombay,3.0,0.0,Maintained,Indian Institute of Technology Bombay
3,INST0583,Indian Institute of Technology Kanpur,4.0,Indian Institute of Technology Kanpur,4.0,0.0,Maintained,Indian Institute of Technology Kanpur
4,INST0584,Indian Institute of Technology Kharagpur,5.0,Indian Institute of Technology Kharagpur,5.0,0.0,Maintained,Indian Institute of Technology Kharagpur


# Q1: Identify the Top 10 Institutions with Improved Rankings

In this question, we identify the institutions that improved their ranking positions between the selected years.

Rank improvement is calculated as:

Rank Improvement = Previous Year Rank - Current Year Rank

A positive value indicates an improvement in ranking position.

For example, if an institution moves from rank 20 to rank 10, its improvement is 10 positions.

##  Identify Institutions with Improved Rankings

In [7]:
def get_top_improved(comparison_data, top_n=10):

    improved = comparison_data[
        comparison_data["rank_improvement"] > 0
    ].copy()

    improved = improved.sort_values(
        "rank_improvement",
        ascending=False
    )

    return improved.head(top_n)


top_improved = get_top_improved(comparison)

display(
    top_improved[
        [
            "institute_name",
            "previous_rank",
            "current_rank",
            "rank_improvement"
        ]
    ]
)

,institute_name,previous_rank,current_rank,rank_improvement
81,Madan Mohan Malaviya University of Technology,84.0,60.0,24.0
82,Indraprastha Institute of Information Technolo...,85.0,63.0,22.0
87,Christ University,93.0,76.0,17.0
33,Indian Institute of Technology Patna,34.0,19.0,15.0
53,Indian Institute of Technology Bhubaneswar,54.0,39.0,15.0
86,"Vignan's Foundation for Science, Technology an...",91.0,80.0,11.0
89,"Jain University, Bangalore",95.0,84.0,11.0
19,Birla Institute of Technology & Science -Pilani,20.0,11.0,9.0
46,International Institute of Information Technol...,47.0,38.0,9.0
52,Saveetha Institute of Medical and Technical Sc...,53.0,45.0,8.0


##  Visualize the Top 10 Improved Institutions

A bar chart displays the number of ranking positions gained by each institution.

In [8]:
if not top_improved.empty:
    plot_data = top_improved.head(10).sort_values("rank_improvement")
    fig = px.bar(plot_data, x="rank_improvement", y="institute_name", orientation="h",
                 hover_data=["previous_rank","current_rank"],
                 title=f"Top rank improvements ({previous_year} → {current_year})",
                 labels={"rank_improvement":"Positions gained","institute_name":"Institution"})
    fig.show()
else:
    print("No exact-to-exact institutions improved.")

# Q2: Identify the Top 10 Institutions with Declining Rankings

In this question, we identify institutions whose ranking positions became worse between the selected years.

A negative rank improvement indicates a decline.

For example, if an institution moves from rank 10 to rank 20, its rank improvement is -10, meaning it declined by 10 positions.

We identify the institutions with the largest ranking declines.

##  Identify Institutions with Declining Rankings

In [9]:
def get_top_declined(comparison_data, top_n=10):

    declined = comparison_data[
        comparison_data["rank_improvement"] < 0
    ].copy()

    declined["positions_declined"] = (
        -declined["rank_improvement"]
    )

    declined = declined.sort_values(
        "positions_declined",
        ascending=False
    )

    return declined.head(top_n)


top_declined = get_top_declined(comparison)

display(
    top_declined[
        [
            "institute_name",
            "previous_rank",
            "current_rank",
            "positions_declined"
        ]
    ]
)

,institute_name,previous_rank,current_rank,positions_declined
62,Defence Institute of Advanced Technology,63.0,92.0,29.0
44,National Institute of Technology Delhi,45.0,65.0,20.0
80,Sri Krishna College of Engineering and Technology,83.0,100.0,17.0
69,National Institute of Technology Raipur,71.0,86.0,15.0
67,National Institute of Technology Meghalaya,68.0,83.0,15.0
75,College of Engineering - Pune,77.0,90.0,13.0
56,Netaji Subhas University of Technology (NSUT),57.0,70.0,13.0
39,National Institute of Technology Silchar,40.0,50.0,10.0
50,Indian Institute of Space Science and Technology,51.0,61.0,10.0
21,Indian Institute of Technology Ropar,22.0,32.0,10.0


##  Visualize the Top 10 Declined Institutions

The bar chart compares the number of ranking positions lost by each institution.

In [10]:
if not top_declined.empty:
    plot_data = top_declined.head(10).sort_values("positions_declined")
    fig = px.bar(plot_data, x="positions_declined", y="institute_name", orientation="h",
                 hover_data=["previous_rank","current_rank"],
                 title=f"Top rank declines ({previous_year} → {current_year})",
                 labels={"positions_declined":"Positions lost","institute_name":"Institution"})
    fig.show()
else:
    print("No exact-to-exact institutions declined.")

# Q3: Percentage of exact-rank movements and rank-band transitions

This summary includes exact-to-exact movements (improved, declined, maintained), institutions that dropped from an exact rank into a published band, and institutions that rose from a published band to an exact rank. The denominator is the total of these classified transitions; unmatched institutions and band-to-band transitions are not counted.


##  Calculate Ranking Movement Counts

In [ ]:
movement_order = ["Improved","Declined","Maintained","Dropped to band","Rose from band"]

status_pairs = previous_data[["institution_key","rank_status"]].rename(
    columns={"rank_status":"previous_status"}
).merge(
    current_data[["institution_key","rank_status"]].rename(
        columns={"rank_status":"current_status"}
    ), on="institution_key", how="inner"
)
status_pairs["previous_status"] = status_pairs["previous_status"].astype(str).str.lower()
status_pairs["current_status"] = status_pairs["current_status"].astype(str).str.lower()

band_counts = {
    "Dropped to band": int(((status_pairs.previous_status=="exact") &
                            (status_pairs.current_status=="band")).sum()),
    "Rose from band": int(((status_pairs.previous_status=="band") &
                           (status_pairs.current_status=="exact")).sum())
}
exact_counts = comparison["rank_movement"].value_counts().to_dict()
movement_counts = {
    "Improved":int(exact_counts.get("Improved",0)),
    "Declined":int(exact_counts.get("Declined",0)),
    "Maintained":int(exact_counts.get("Maintained",0)),
    **band_counts
}
movement_summary = pd.DataFrame({
    "Movement":movement_order,
    "Number of Institutions":[movement_counts[x] for x in movement_order]
})
total_compared = int(movement_summary["Number of Institutions"].sum())
movement_summary["Percentage"] = (
    movement_summary["Number of Institutions"]/total_compared*100 if total_compared else 0.0
)
display(movement_summary)
print("Includes exact-to-exact movements and exact↔band transitions. Band-to-band and unmatched records are excluded.")


## Visualize Ranking Movement Percentages

A bar chart helps us compare the percentage of institutions that improved, declined, or maintained their rankings.

In [ ]:
if total_compared > 0:
    fig = px.bar(
        movement_summary, x="Movement", y="Percentage", color="Movement",
        text=movement_summary["Percentage"].map(lambda v:f"{v:.1f}%"),
        hover_data={"Number of Institutions":True,"Percentage":":.1f"},
        title=f"Exact-rank movement and rank-band transitions ({previous_year} → {current_year})",
        labels={"Percentage":"Share of classified transitions (%)"}
    )
    fig.update_yaxes(range=[0,100])
    fig.show()
else:
    print("No classifiable exact-rank or exact↔band transitions for this period.")


# Q4: Top-10 entries and exits across all available years

A single pair of years may legitimately have zero entries and exits, making a two-bar chart uninformative. This revised question tracks top-10 membership changes year by year across all available years for the selected category.

##  Identify Top 10 Institutions in Both Years

In [ ]:
def top10_entry_exit_timeline(data, category):
    subset = data[data["category"].eq(category)].copy()
    subset["rank"] = pd.to_numeric(subset["rank"], errors="coerce")
    subset = subset[subset["rank_status"].astype(str).str.strip().str.lower().eq("exact")]
    subset = subset.dropna(subset=["institution_key","rank"]).drop_duplicates(["year","institution_key"])
    years = sorted(subset["year"].dropna().unique())
    previous, previous_names, rows = set(), {}, []
    for year in years:
        current_rows = subset[(subset["year"] == year) & (subset["rank"] <= 10)]
        current = set(current_rows["institution_key"])
        current_names = dict(zip(current_rows["institution_key"], current_rows["institute_name"]))
        entered, left = sorted(current-previous), sorted(previous-current)
        rows.append({
            "year":year, "Membership changes":len(entered)+len(left),
            "Entered names":", ".join(current_names.get(k,str(k)) for k in entered) or "None",
            "Left names":", ".join(previous_names.get(k,str(k)) for k in left) or "None",
            "Top 10 Institutions":len(current)
        })
        previous, previous_names = current, current_names
    return pd.DataFrame(rows)

top10_timeline = top10_entry_exit_timeline(df, selected_category)
display(top10_timeline)


##  Find Institutions Entering and Leaving the Top 10

In [ ]:
fig = px.bar(
    top10_timeline, x="year", y="Membership changes",
    hover_data=["Entered names","Left names","Top 10 Institutions"],
    title=f"Top-10 membership changes over time – {selected_category}",
    labels={"year":"Year","Membership changes":"Institutions entering or leaving"}
)
fig.show()


## Visualize Top 10 Entries and Exits

We compare the number of institutions entering and leaving the top 10 ranking positions.

In [15]:
# The multi-year timeline above replaces the previous single-pair two-zero-bar chart.

# Q5: Analyse transitions between published ranking groups

Exact rank records are grouped into Top 10, 11–50 and 51–100. Published band records retain their actual bands (for example, 101–150 and 151–200). Institutions without a rank/band in one of the selected years are classified as Not ranked / not in published band when matching against the category's union of institution keys. Band records are not assigned fabricated exact ranks.

##  Create Ranking Groups

We create a reusable function that assigns an institution to a ranking group based on its exact rank.

In [ ]:
def exact_rank_group(rank):
    if pd.isna(rank):
        return "Not ranked / not in published band"
    if rank <= 10:
        return "Top 10"
    if rank <= 50:
        return "11-50"
    if rank <= 100:
        return "51-100"
    if rank <= 200:
        return "101-200 (exact)"
    return "Outside published exact ranks"

def build_band_aware(data):
    out = data.copy()
    exact = out["rank_status"].astype(str).str.strip().str.lower().eq("exact")
    out["transition_group"] = out["rank_band_label"].where(~exact)
    out.loc[exact, "transition_group"] = out.loc[exact, "rank"].apply(exact_rank_group)
    out["transition_group"] = out["transition_group"].fillna("Not ranked / not in published band")
    return out[["institution_key","institute_name","transition_group","rank_status","rank","rank_low","rank_high"]]

previous_group_data = build_band_aware(previous_data).rename(columns={"transition_group":"Previous Group"})
current_group_data = build_band_aware(current_data).rename(columns={"transition_group":"Current Group"})
group_comparison = previous_group_data[["institution_key","Previous Group"]].merge(
    current_group_data[["institution_key","Current Group"]], on="institution_key", how="outer"
).fillna({"Previous Group":"Not ranked / not in published band",
          "Current Group":"Not ranked / not in published band"})

observed_groups = set(group_comparison["Previous Group"]) | set(group_comparison["Current Group"])
preferred_order = ["Top 10","11-50","51-100","101-200 (exact)"]
rank_group_labels = [g for g in preferred_order if g in observed_groups]
rank_group_labels += sorted(g for g in observed_groups if g not in preferred_order and
                             g != "Not ranked / not in published band")
if "Not ranked / not in published band" in observed_groups:
    rank_group_labels.append("Not ranked / not in published band")


##  Match Ranking Groups Across Years

We match the same institutions using institution_key and compare their previous and current ranking groups.

In [17]:
# Add all institutions found in either selected year, preserving missing-side status.
print("Institutions represented in band-aware transition analysis:", len(group_comparison))
display(group_comparison.head())

Institutions represented in band-aware transition analysis: 210


,institution_key,Previous Group,Current Group
0,INST0012,51-100,51-100
1,INST0039,11-50,11-50
2,INST0055,11-50,11-50
3,INST0058,101-150,101-150
4,INST0060,101-150,101-150


##  Create the Ranking Transition Table

A transition table shows how many institutions moved from each previous ranking group to each current ranking group.

Rows represent the previous year's ranking groups.

Columns represent the current year's ranking groups.

In [ ]:
transition_table = pd.crosstab(group_comparison["Previous Group"], group_comparison["Current Group"])
transition_table = transition_table.reindex(
    index=[g for g in rank_group_labels if g in transition_table.index],
    columns=[g for g in rank_group_labels if g in transition_table.columns],
    fill_value=0
)
transition_table = transition_table.loc[
    transition_table.sum(axis=1)>0, transition_table.sum(axis=0)>0
]
display(transition_table)

row_totals = transition_table.sum(axis=1).replace(0, np.nan)
transition_percent = transition_table.div(row_totals, axis=0).mul(100).fillna(0)
print("Row percentages (each previous-year group sums to 100% where data exist):")
display(transition_percent.round(1))


##  Visualize Ranking Group Transitions

A heatmap provides a visual representation of movement between ranking groups.

Higher values indicate that more institutions moved from one particular previous group to the corresponding current group.

In [ ]:
fig = px.imshow(
    transition_table, text_auto=True, aspect="auto", color_continuous_scale="Purples",
    title=f"Band-aware ranking transitions ({previous_year} → {current_year})",
    labels={"x":"Current-year group","y":"Previous-year group","color":"Institutions"},
    x=transition_table.columns, y=transition_table.index
)
# Use dark annotation text so low-value cells remain legible on pale fills.
fig.update_traces(textfont={"color":"black"})
fig.update_layout(width=max(850,110*len(transition_table.columns)),
                  height=max(550,65*len(transition_table.index)),
                  margin={"l":150,"b":140})
fig.update_xaxes(tickangle=-35)
fig.show()

fig_pct = px.imshow(
    transition_percent, text_auto=".1f", aspect="auto", color_continuous_scale="Blues",
    title="Row percentages for ranking-group transitions",
    labels={"x":"Current-year group","y":"Previous-year group","color":"Row %"},
    x=transition_percent.columns, y=transition_percent.index
)
fig_pct.update_traces(textfont={"color":"black"})
fig_pct.update_layout(width=max(850,110*len(transition_percent.columns)),
                      height=max(550,65*len(transition_percent.index)),
                      margin={"l":150,"b":140})
fig_pct.update_xaxes(tickangle=-35)
fig_pct.show()


# Final Summary – Section 7

In this section, we analysed institutional ranking transitions and mobility using the NIRF dataset.

The analysis covered five major questions:

1. Identified the top 10 institutions that improved their ranking positions.
2. Identified the top 10 institutions whose rankings declined.
3. Calculated the percentages of institutions that improved, declined, or maintained their ranks.
4. Identified institutions entering and leaving the top 10.
5. Analysed transitions between different ranking groups.

Institutions were matched across years using institution_key.

Only institutions with exact ranks were considered for year-to-year comparisons.

The analysis helps us understand changes in institutional rankings, identify major improvements and declines, and observe movement across ranking groups.

Overall, ranking transition analysis provides a better understanding of institutional performance over time.

In [20]:
print("="*65)
print("SECTION 7 – RANKING TRANSITIONS AND INSTITUTIONAL MOBILITY (REVISED)")
print("="*65)
print("Category:", selected_category, "| Years:", previous_year, "→", current_year)
print("Exact-to-exact matched institutions:", len(comparison))
print("Movement percentages (exact ranks only):")
display(movement_summary)
print("Previous-only institutions:", len(unmatched_previous))
print("Current-only institutions:", len(unmatched_current))
print("\nTop-10 membership timeline:")
display(top10_timeline)
print("\nBand-aware transition counts:")
display(transition_table)
print("\nBand-aware transition row percentages:")
display(transition_percent.round(1))
print("\nCaution: rank-band midpoints are estimates, not exact ranks.")

SECTION 7 – RANKING TRANSITIONS AND INSTITUTIONAL MOBILITY (REVISED)
Category: Engineering | Years: 2024 → 2025
Exact-to-exact matched institutions: 93
Movement percentages (exact ranks only):


,Movement,Number of Institutions,Percentage
0,Improved,36,38.709677
1,Declined,38,40.860215
2,Maintained,19,20.430108


Previous-only institutions: 9
Current-only institutions: 10

Top-10 membership timeline:


,year,Entered Top 10,Left Top 10,Top 10 Institutions
0,2017,0,0,10
1,2018,1,1,10
2,2019,1,1,10
3,2020,1,1,10
4,2021,1,1,10
5,2022,0,0,10
6,2023,1,1,10
7,2024,1,1,10
8,2025,0,0,10



Band-aware transition counts:


Current Group,Top 10,11-50,51-100,101-150,151-200,Outside published exact ranks,Not ranked / not in published band
Previous Group,,,,,,,
Top 10,10,0,0,0,0,0,0
11-50,0,37,3,0,0,0,0
51-100,0,2,41,6,1,0,0
101-150,0,0,6,35,9,0,0
151-200,0,0,0,7,34,0,9
Outside published exact ranks,0,0,0,0,0,0,0
Not ranked / not in published band,0,1,0,2,7,0,0



Band-aware transition row percentages:


Current Group,Top 10,11-50,51-100,101-150,151-200,Outside published exact ranks,Not ranked / not in published band
Previous Group,,,,,,,
Top 10,100.0,0.0,0.0,0.0,0.0,0.0,0.0
11-50,0.0,92.5,7.5,0.0,0.0,0.0,0.0
51-100,0.0,4.0,82.0,12.0,2.0,0.0,0.0
101-150,0.0,0.0,12.0,70.0,18.0,0.0,0.0
151-200,0.0,0.0,0.0,14.0,68.0,0.0,18.0
Outside published exact ranks,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Not ranked / not in published band,0.0,10.0,0.0,20.0,70.0,0.0,0.0



Caution: rank-band midpoints are estimates, not exact ranks.
